# 🌾 Kisan Saathi — Unified Multi-Dataset Model Training
This notebook trains a production-grade crop disease neural network by combining:
1. **PlantVillage Dataset** (54,300+ laboratory leaf images across 38 classes)
2. **PlantDoc Dataset (IIT Delhi & Cornell)** (2,500+ real-field Indian farm smartphone photos)
3. **Negative Background Class** (Soil, hands, tools, neutral surfaces to prevent false positive detections)

### Architecture
- **Backbone**: `MobileNetV3-Large` (pretrained on ImageNet)
- **Augmentations**: ColorJitter, RandomAffine, RandomRotation, RandomErasing
- **Loss & Optimizer**: Label Smoothing CrossEntropy + AdamW + Cosine Annealing Scheduler

### Step 1: Download PlantVillage & PlantDoc Datasets

In [ ]:
# Clone PlantVillage & PlantDoc datasets
!git clone --depth 1 https://github.com/spMohanty/PlantVillage-Dataset.git
!git clone --depth 1 https://github.com/pratikkayal/PlantDoc-Dataset.git
print('✅ Both datasets downloaded successfully!')

### Step 2: Merge & Standardize Datasets into `unified_dataset/`

In [ ]:
import os
import shutil
from PIL import Image
import numpy as np

UNIFIED_DIR = 'unified_dataset'
os.makedirs(UNIFIED_DIR, exist_ok=True)

# 1. Copy PlantVillage images
pv_source = 'PlantVillage-Dataset/raw/color'
print('Merging PlantVillage images...')
for cls_name in os.listdir(pv_source):
    src_cls_dir = os.path.join(pv_source, cls_name)
    if os.path.isdir(src_cls_dir):
        dest_cls_dir = os.path.join(UNIFIED_DIR, cls_name)
        os.makedirs(dest_cls_dir, exist_ok=True)
        for fname in os.listdir(src_cls_dir):
            if fname.lower().endswith(('.jpg', '.jpeg', '.png')):
                shutil.copy2(os.path.join(src_cls_dir, fname), os.path.join(dest_cls_dir, f'pv_{fname}'))

# 2. Map & Copy PlantDoc in-field smartphone images
PLANTDOC_TO_PLANTVILLAGE = {
    'Apple Scab Leaf': 'Apple___Apple_scab',
    'Apple leaf': 'Apple___healthy',
    'Apple rust leaf': 'Apple___Cedar_apple_rust',
    'Bell_pepper leaf spot': 'Pepper,_bell___Bacterial_spot',
    'Bell_pepper leaf': 'Pepper,_bell___healthy',
    'Blueberry leaf': 'Blueberry___healthy',
    'Cherry leaf': 'Cherry_(including_sour)___healthy',
    'Corn Gray leaf spot': 'Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot',
    'Corn leaf blight': 'Corn_(maize)___Northern_Leaf_Blight',
    'Corn rust leaf': 'Corn_(maize)___Common_rust_',
    'Grape leaf black rot': 'Grape___Black_rot',
    'Grape leaf': 'Grape___healthy',
    'Peach leaf': 'Peach___healthy',
    'Potato leaf early blight': 'Potato___Early_blight',
    'Potato leaf late blight': 'Potato___Late_blight',
    'Potato leaf': 'Potato___healthy',
    'Raspberry leaf': 'Raspberry___healthy',
    'Soyabean leaf': 'Soybean___healthy',
    'Squash Powdery mildew leaf': 'Squash___Powdery_mildew',
    'Strawberry leaf': 'Strawberry___healthy',
    'Tomato Early blight leaf': 'Tomato___Early_blight',
    'Tomato Septoria leaf spot': 'Tomato___Septoria_leaf_spot',
    'Tomato leaf bacterial spot': 'Tomato___Bacterial_spot',
    'Tomato leaf late blight': 'Tomato___Late_blight',
    'Tomato leaf mosaic virus': 'Tomato___Tomato_mosaic_virus',
    'Tomato leaf yellow virus': 'Tomato___Tomato_Yellow_Leaf_Curl_Virus',
    'Tomato leaf': 'Tomato___healthy',
    'Tomato mold leaf': 'Tomato___Leaf_Mold'
}

print('Merging PlantDoc in-field smartphone photos...')
for split in ['train', 'test']:
    doc_path = os.path.join('PlantDoc-Dataset', split)
    if os.path.exists(doc_path):
        for doc_cls in os.listdir(doc_path):
            mapped_target = PLANTDOC_TO_PLANTVILLAGE.get(doc_cls)
            if mapped_target:
                dest_cls_dir = os.path.join(UNIFIED_DIR, mapped_target)
                os.makedirs(dest_cls_dir, exist_ok=True)
                doc_src_dir = os.path.join(doc_path, doc_cls)
                for fname in os.listdir(doc_src_dir):
                    if fname.lower().endswith(('.jpg', '.jpeg', '.png')):
                        shutil.copy2(os.path.join(doc_src_dir, fname), os.path.join(dest_cls_dir, f'doc_{split}_{fname}'))

# 3. Add Negative / Background Class (Soil, Hands, Non-plants)
bg_dir = os.path.join(UNIFIED_DIR, 'Background_Without_Leaves')
os.makedirs(bg_dir, exist_ok=True)
for i in range(100):
    # Generate neutral & soil textured images
    color_base = np.random.choice(['soil', 'wall', 'skin', 'gray'])
    if color_base == 'soil':
        base_rgb = np.random.randint(60, 110, size=(224, 224, 3), dtype=np.uint8)
    elif color_base == 'wall':
        base_rgb = np.random.randint(180, 240, size=(224, 224, 3), dtype=np.uint8)
    elif color_base == 'skin':
        base_rgb = np.array([210, 160, 140], dtype=np.uint8) + np.random.randint(-15, 15, size=(224, 224, 3), dtype=np.int16).clip(0, 255).astype(np.uint8)
    else:
        base_rgb = np.random.randint(90, 160, size=(224, 224, 3), dtype=np.uint8)
    Image.fromarray(base_rgb).save(os.path.join(bg_dir, f'synth_bg_{i}.jpg'))

total_classes = len(os.listdir(UNIFIED_DIR))
total_images = sum([len(os.listdir(os.path.join(UNIFIED_DIR, d))) for d in os.listdir(UNIFIED_DIR)])
print(f'✅ Unified Dataset Ready: {total_images} images across {total_classes} classes.')

### Step 3: Train MobileNetV3-Large with PyTorch on GPU

In [ ]:
import json
import torch
import torchvision
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split
import torch.nn as nn
import torch.optim as optim
from torchvision.models import mobilenet_v3_large, MobileNet_V3_Large_Weights

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using Compute Device: {device} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"})')

# Heavy Field Augmentation Pipeline
train_transform = transforms.Compose([
    transforms.Resize((240, 240)),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.3),
    transforms.RandomRotation(degrees=25),
    transforms.ColorJitter(brightness=0.25, contrast=0.25, saturation=0.25, hue=0.08),
    transforms.RandomAffine(degrees=15, translate=(0.1, 0.1), scale=(0.85, 1.15)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.2, scale=(0.02, 0.2))
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

full_dataset = datasets.ImageFolder(root=UNIFIED_DIR, transform=train_transform)
num_classes = len(full_dataset.classes)

train_size = int(0.85 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)

# Build Model with MobileNetV3-Large
model = mobilenet_v3_large(weights=MobileNet_V3_Large_Weights.DEFAULT)
in_features = model.classifier[0].in_features
model.classifier = nn.Sequential(
    nn.Linear(in_features, 1024),
    nn.Hardswish(),
    nn.Dropout(p=0.3),
    nn.Linear(1024, num_classes)
)
model = model.to(device)

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = optim.AdamW(model.parameters(), lr=0.0005, weight_decay=1e-4)
epochs = 15
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

best_acc = 0.0
print('Starting Training with Multi-Dataset & Cosine Annealing...')
for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    for step, (inputs, labels) in enumerate(train_loader):
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        
    scheduler.step()
    
    # Validation
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (preds == labels).sum().item()
            
    val_acc = 100.0 * correct / total
    print(f'Epoch [{epoch+1}/{epochs}] Validation Accuracy: {val_acc:.2f}% (LR: {scheduler.get_last_lr()[0]:.6f})')
    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), 'best_model.pth')

print(f'\n🎉 Training Complete! Best Validation Accuracy: {best_acc:.2f}%')

### Step 4: Export to ONNX & Download Files

In [ ]:
# Export to ONNX
model.load_state_dict(torch.load('best_model.pth'))
model.eval()

dummy_input = torch.randn(1, 3, 224, 224, device=device)
torch.onnx.export(
    model,
    dummy_input,
    'crop_disease_model.onnx',
    export_params=True,
    opset_version=11,
    do_constant_folding=True,
    input_names=['input'],
    output_names=['output'],
    dynamic_axes={'input': {0: 'batch_size'}, 'output': {0: 'batch_size'}}
)

with open('classes.json', 'w') as f:
    json.dump(full_dataset.classes, f, indent=2)

print('✅ Saved crop_disease_model.onnx and classes.json successfully!')

# Automatic download for Colab
try:
    from google.colab import files
    files.download('crop_disease_model.onnx')
    files.download('classes.json')
except Exception as e:
    print('Download manually from files pane on the left:', e)